# RG3 가스 선정 후보 및 동봉 데이터 검증
기존 가스17/정상566 기준. 탐색 후보 성적이며 독립검증으로 해석하지 않는다.

In [1]:
from pathlib import Path
import hashlib, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
rg=d[d['product'].eq('RG3')].copy()
assert len(rg)==591 and rg.TimeStamp.is_unique
assert rg.PassOrFail.eq(0).sum()==566
assert rg.Reason.str.contains('가스',na=False).sum()==17
assert rg.Reason.str.contains('미성형',na=False).sum()==8
rh=pd.read_csv(root/'data/audit/rg3_rh_labeled_with_timestamp.csv',parse_dates=['TimeStamp'])
assert len(rh)==591 and set(rh.TimeStamp)==set(rg.TimeStamp)
expected=rg[rg.PassOrFail.eq(0)|rg.Reason.eq('가스')].set_index('TimeStamp').PassOrFail
rows=[]
settings=[
 ('22_RF_baseline','rg3_nested_ensemble_predictions.csv',{'scheme':'leave_one_block_out','model':'max_union'},'predicted_train_fpr1',(4,6,13,560)),
 ('47_innovation_seed_mean','rg3_innovation_seed_sensitivity_predictions.csv',{'scheme':'leave_one_block_out','model':'three_seed_mean'},'prediction',(5,6,12,560)),
 ('60_temperature_majority_pressure','rg3_pressure_seed_majority_predictions.csv',{'scheme':'leave_one_block_out','anchor':'majority57'},'prediction',(7,6,10,560)),
 ('60_thermal_anchor_pressure','rg3_pressure_seed_majority_predictions.csv',{'scheme':'leave_one_block_out','anchor':'exploratory54'},'prediction',(8,6,9,560))]
for name,file,filters,column,answer in settings:
 p=pd.read_csv(t/file,parse_dates=['TimeStamp'])
 for key,value in filters.items():p=p[p[key].eq(value)]
 assert len(p)==583 and p.TimeStamp.is_unique and set(p.TimeStamp)==set(expected.index)
 assert np.array_equal(p.actual.to_numpy(),expected.loc[p.TimeStamp].to_numpy())
 tn,fp,fn,tp=confusion_matrix(p.actual,p[column].astype(int),labels=[0,1]).ravel()
 assert (tp,fp,fn,tn)==answer
 rows.append(dict(candidate=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),precision=tp/(tp+fp),recall=tp/(tp+fn),evaluation='development_leave_one_block_out'))
summary=pd.DataFrame(rows)
summary.to_csv(t/'rg3_selected_candidates_verified.csv',index=False,encoding='utf-8-sig')
files=['data/original_split/moldset_labeled_rg3.csv','data/original_split/moldset_unlabeled_rg3.csv','data/audit/rg3_rh_labeled_with_timestamp.csv','reports/tables/original_labeled_shot_features.csv']
manifest=[]
for relative in files:
 p=root/relative; raw=p.read_bytes();n=len(pd.read_csv(p))
 manifest.append(dict(path=relative,rows=n,bytes=len(raw),sha256=hashlib.sha256(raw).hexdigest()))
pd.DataFrame(manifest).to_csv(t/'rg3_selected_input_manifest.csv',index=False,encoding='utf-8-sig')
print(summary.to_string(index=False))
print(pd.DataFrame(manifest).to_string(index=False))
print('Verified saved predictions and source cohort; models were NOT retrained by this packaging audit.')


                       candidate  TP  FP  FN  TN       F1  precision   recall                      evaluation
                  22_RF_baseline   4   6  13 560 0.296296   0.400000 0.235294 development_leave_one_block_out
         47_innovation_seed_mean   5   6  12 560 0.357143   0.454545 0.294118 development_leave_one_block_out
60_temperature_majority_pressure   7   6  10 560 0.466667   0.538462 0.411765 development_leave_one_block_out
      60_thermal_anchor_pressure   8   6   9 560 0.516129   0.571429 0.470588 development_leave_one_block_out
                                             path  rows    bytes                                                           sha256
      data/original_split/moldset_labeled_rg3.csv  1182   687519 bfea073c3d3590e71df738667f0c2d5bde5be9ffab06e2f5d49b3a7f589cc457
    data/original_split/moldset_unlabeled_rg3.csv 72484 21732005 91d61682076821601fae488dbe72864af493b2b5f39603d5a33003f77cf71238
     data/audit/rg3_rh_labeled_with_timestamp.csv   591   35